# 🔬 Notebook 3: Shopping Cart — Deep Dive

## 🛠️ Setup

```bash
cd 06-system-designs/shopping-cart
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Deep dive 1

### Inventory reservation with TTL

Two problems:
1. If we decrement stock at `Add to cart`, idle carts block real buyers.
2. If we decrement only at `Pay`, two buyers can both succeed the stock check.

**Fix:** at checkout start, create a reservation with a TTL (say 10 min). If payment succeeds, convert reservation → sale. If TTL expires, reservation auto-releases.

In [ ]:
import time, threading

class Inventory:
    def __init__(self, stock):
        self.stock = dict(stock)           # sku → available
        self.reservations = {}              # id → (sku, qty, expires_at)
        self.lock = threading.Lock()

    def reserve(self, rid, sku, qty, ttl=10):
        with self.lock:
            self._sweep()
            if self.stock.get(sku, 0) < qty:
                return False
            self.stock[sku] -= qty
            self.reservations[rid] = (sku, qty, time.time()+ttl)
            return True

    def confirm(self, rid):
        with self.lock:
            if rid in self.reservations:
                del self.reservations[rid]  # stock stays decremented
                return True
            return False

    def _sweep(self):
        now = time.time()
        expired = [k for k,(s,q,exp) in self.reservations.items() if exp < now]
        for k in expired:
            s,q,_ = self.reservations.pop(k)
            self.stock[s] += q             # give back

inv = Inventory({"A": 2})
print("alice reserves 1:", inv.reserve("r1", "A", 1, ttl=1))
print("bob reserves 2:", inv.reserve("r2", "A", 2))   # only 1 left → False
print("bob reserves 1:", inv.reserve("r3", "A", 1))
print("stock now:", inv.stock)
time.sleep(1.1)
inv._sweep()
print("after alice TTL:", inv.stock)

## Deep dive 2

### Checkout saga

Checkout touches multiple services: inventory, payment, order. We can't put them in one DB transaction, so we use a **saga** with compensations:

```
reserve stock ──► charge card ──► create order
    │(fail)          │(fail)
    ▼                ▼
  abort          release stock
```

In [ ]:
class FakeCard:
    def __init__(self, ok=True): self.ok = ok
    def charge(self, amount):
        if not self.ok: raise RuntimeError("card declined")
        return {"txn": "tx_" + str(amount)}

def checkout(inv, card, cart_id, sku, qty, amount):
    rid = f"res-{cart_id}"
    if not inv.reserve(rid, sku, qty, ttl=60):
        return {"status": "out_of_stock"}
    try:
        txn = card.charge(amount)
    except Exception as e:
        # compensate
        inv._sweep(); inv.stock[sku] += qty; inv.reservations.pop(rid, None)
        return {"status": "payment_failed", "error": str(e)}
    inv.confirm(rid)
    return {"status": "paid", "txn": txn}

inv = Inventory({"A": 5})
print(checkout(inv, FakeCard(ok=True),  "c1", "A", 2, 20))
print(checkout(inv, FakeCard(ok=False), "c2", "A", 1, 10))
print("stock after:", inv.stock)

## Closing thoughts

- Split **state** (cart) from **inventory** (scarce resource).
- Use **reservations with TTL** — not permanent decrements.
- Use **idempotency keys** on money-moving endpoints.
- Model checkout as a **saga** so partial failures are recoverable.